In [1]:
from pathlib import Path
import numpy as np

COND_DIR = Path("SNP+MET+CNV+RNA__attention__morgan__mixed__c94ea3")
DRUG = "5-Fluorouracil"
N_FOLDS = 5

In [ ]:
# Ensemble result (single drug, averaged over folds)
ensemble_shap = np.load(COND_DIR / "all_drugs_ensemble_shap.npz")
ensemble_X = np.load(COND_DIR / "all_drugs_ensemble_X.npz")

shap_vals = ensemble_shap[DRUG]   # [n_cells, 4148]
X = ensemble_X[DRUG]              # [n_cells, 4148]
shap_vals.shape, X.shape

# Per-fold result (same drug, one fold)
fold1 = np.load(COND_DIR / "fold_1" / f"{DRUG}.npz")
fold1["shap"].shape, fold1["X"].shape

((872, 4148), (872, 4148))

In [ ]:
# Stack all folds for one drug, e.g. to look at per-fold variance
per_fold_shap = np.stack([
    np.load(COND_DIR / f"fold_{i}" / f"{DRUG}.npz")["shap"]
    for i in range(1, N_FOLDS + 1)
])  # [n_folds, n_cells, 4148]

per_fold_shap.mean(axis=0)  # should match all_drugs_ensemble_shap.npz[DRUG]
np.allclose(per_fold_shap.mean(axis=0), shap_vals, atol=1e-5)

True

In [ ]:
# Split a drug's flattened SHAP back into fingerprint + gene x omics
N_GENE, N_OMICS = 909, 4

fp_shap = shap_vals[:, :512]
gene_shap = shap_vals[:, 512:].reshape(-1, N_GENE, N_OMICS)  # [n_cells, gene, omics]
fp_shap.shape, gene_shap.shape

((872, 512), (872, 909, 4))